### Esquema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark.sql("USE dbestudiantes")


layer = "Bronze"

### Función auxiliar para registrar metadatos y logs

In [0]:
%run "./00_utils_logging_functions"

### Funciones utiles para el proyecto

In [0]:
from pyspark.sql.functions import col

lista_tablas = ["raw_students", "raw_enrollments", "raw_subjects"]

bronze_schemas = {
    "raw_students": [
        col("id_estudiante").cast("int"),
        col("nombre").cast("string"),
        col("apellido").cast("string"),
        col("grado").cast("int"),
        col("edad").cast("int"),
        col("repitente").cast("string"),
        col("alumno_ayudante").cast("string"),
        col("id_ingestion").cast("int"),
        col("timestamp_ingestion").cast("timestamp")
    ],
    "raw_enrollments": [
        col("id_estudiante").cast("int"),
        col("id_materia").cast("int"),
        col("nota").cast("double"),
        col("fecha").cast("string"),
        col("id_ingestion").cast("int"),
        col("timestamp_ingestion").cast("timestamp")
    ],
    "raw_subjects": [
        col("id_materia").cast("int"),
        col("nombre_materia").cast("string"),
        col("creditos").cast("int"),
        col("id_ingestion").cast("int"),
        col("timestamp_ingestion").cast("timestamp")
    ]
}

for t in lista_tablas:

    job_name = f"bronze_{t}"
    bronze_table = f"workspace.dbestudiantes.bronze_{t}"
    landing_table = f"workspace.dbestudiantes.landing_{t}"

    try:
        print(f"Procesando tabla: {t}")
        log_event(job_name, layer, "INFO", f"Iniciando procesamiento de {t}")

        df_landing = spark.table(landing_table)

        # Obtener último id_ingestion
        if spark.catalog.tableExists(bronze_table):
            df_bronze = spark.table(bronze_table)
            ultimo_id = df_bronze.agg({"id_ingestion": "max"}).collect()[0][0]
        else:
            df_bronze = None
            ultimo_id = 0

        # Filtrar nuevos registros
        df_nuevos = df_landing.filter(col("id_ingestion") > ultimo_id)
        row_count = df_nuevos.count()

        # Aplicar esquema Bronze
        df_bronze_ready = df_nuevos.select(*bronze_schemas[t])

        # Escribir en Bronze
        if df_bronze is None:
            df_bronze_ready.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
            log_event(job_name, layer, "INFO", f"Tabla Bronze {t} creada.")
        else:
            df_bronze_ready.write.format("delta").mode("append").saveAsTable(bronze_table)
            log_event(job_name, layer, "INFO", f"Tabla Bronze {t} actualizada.")

        # Registrar metadatos
        if row_count > 0:
            ultimo_id_nuevo = df_bronze_ready.agg({"id_ingestion": "max"}).collect()[0][0]
        else:
            ultimo_id_nuevo = ultimo_id

        log_meta(job_name, layer, row_count, ultimo_id_nuevo, "SUCCESS")

        print(f"✔️ Bronze actualizado para {t}")

    except Exception as e:
        log_event(job_name, layer, "ERROR", f"Error procesando {t}: {e}")
        log_meta(job_name, layer, -1, ultimo_id, "ERROR")
        raise




Procesando tabla: raw_students
✔️ Bronze actualizado para raw_students
Procesando tabla: raw_enrollments
✔️ Bronze actualizado para raw_enrollments
Procesando tabla: raw_subjects
✔️ Bronze actualizado para raw_subjects
